In [ ]:
import os
import rasterio
from rasterio.plot import show
from rasterio.mask import mask
import rasterstats
from rasterstats import zonal_stats

import pandas as pd
import geopandas as gpd
import utils
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_root = '../data/inequality_data_grdi'
grdi_src = rasterio.open(os.path.join(data_root, 'povmap-grdi-v1.tif'))
grdi_BUILT = rasterio.open(os.path.join(data_root, 'povmap-grdi-v1_BUILT.tif'))
grdi_CDR = rasterio.open(os.path.join(data_root, 'povmap-grdi-v1_CDR.tif'))
grdi_IMR = rasterio.open(os.path.join(data_root, 'povmap-grdi-v1_IMR.tif'))
grdi_SHDI = rasterio.open(os.path.join(data_root, 'povmap-grdi-v1_SHDI.tif'))
grdi_VNL_2020 = rasterio.open(os.path.join(data_root, 'povmap-grdi-v1_VNL-2020.tif'))
grdi_VNL_slope = rasterio.open(os.path.join(data_root, 'povmap-grdi-v1_VNL-slope.tif'))

In [ ]:
def apply_grdi_avg(row):
    geom = row['geometry']
    # Create mask from polygon
    masked_data, masked_transform = mask(grdi_src, [geom], crop=True, nodata=np.nan)
    return np.nanmean(masked_data)

def apply_grdi_BUILT_avg(row):
    geom = row['geometry']
    masked_data, masked_transform = mask(grdi_BUILT, [geom], crop=True, nodata=np.nan)
    return np.nanmean(masked_data)

def apply_grdi_CDR_avg(row):
    geom = row['geometry']
    masked_data, masked_transform = mask(grdi_CDR, [geom], crop=True, nodata=np.nan)
    return np.nanmean(masked_data)

def apply_grdi_IMR_avg(row):
    geom = row['geometry']
    masked_data, masked_transform = mask(grdi_IMR, [geom], crop=True, nodata=np.nan)
    return np.nanmean(masked_data)

def apply_grdi_SHDI_avg(row):
    geom = row['geometry']
    masked_data, masked_transform = mask(grdi_SHDI, [geom], crop=True, nodata=np.nan)
    return np.nanmean(masked_data)

def apply_grdi_VNL_2020_avg(row):
    geom = row['geometry']
    masked_data, masked_transform = mask(grdi_VNL_2020, [geom], crop=True, nodata=np.nan)
    return np.nanmean(masked_data)

def apply_grdi_VNL_slope_avg(row):
    geom = row['geometry']
    masked_data, masked_transform = mask(grdi_VNL_slope, [geom], crop=True, nodata=np.nan)
    return np.nanmean(masked_data)

In [ ]:
def read_clustering_results(dis_no, n_clusters=2, save_root='../outputs'):
    # Read the clustering results
    clustering_res = pd.read_csv(os.path.join(save_root, dis_no, f'cluster_{n_clusters}','clustering_result.csv'))

    clustering_res = clustering_res[['index', 'label']]
    clustering_res['geometry'] = clustering_res['index'].apply(lambda x: utils.geohash_to_polygon(x))
    clustering_res = gpd.GeoDataFrame(clustering_res, geometry='geometry', crs='EPSG:4326')

    clustering_res['grdi_avg'] = clustering_res.apply(apply_grdi_avg, axis=1)
    clustering_res['grdi_BUILT_avg'] = clustering_res.apply(apply_grdi_BUILT_avg, axis=1)
    clustering_res['grdi_CDR_avg'] = clustering_res.apply(apply_grdi_CDR_avg, axis=1)
    clustering_res['grdi_IMR_avg'] = clustering_res.apply(apply_grdi_IMR_avg, axis=1)
    clustering_res['grdi_SHDI_avg'] = clustering_res.apply(apply_grdi_SHDI_avg, axis=1)
    clustering_res['grdi_VNL_2020_avg'] = clustering_res.apply(apply_grdi_VNL_2020_avg, axis=1)
    clustering_res['grdi_VNL_slope_avg'] = clustering_res.apply(apply_grdi_VNL_slope_avg, axis=1)

    return clustering_res

In [ ]:
def viz_clustering_results(clustering_res, dis_no, n_clusters, save_root='../outputs'):
    fig, ax = plt.subplots(1, 3, figsize=(18, 8))
    fig.suptitle(dis_no, fontsize=20)
    colors = ['crimson', 'darkblue', 'grey', 'dodgerblue']
    color_mapping = {label: color for label, color in enumerate(colors)}
    clustering_res['color'] = clustering_res['label'].map(color_mapping)

    clustering_res.plot(column='label', ax=ax[0], legend=True, color=clustering_res['color'])
    ax[0].set_title('Cluster Label Map', fontsize=15)
    ax[0].tick_params(axis='both', which='major', labelsize=15)
    
    clustering_res.plot(column='grdi_avg', ax=ax[1], legend=True)
    ax[1].set_title('GRDI Average Map', fontsize=15)
    ax[1].tick_params(axis='both', which='major', labelsize=15)

    # plot boxplot for each label
    sns.boxplot(x='label', y='grdi_avg', data=clustering_res, ax=ax[2], palette=colors, hue='label')
    ax[2].set_title('GRDI Average by Cluster Label', fontsize=15)
    ax[2].set_xlabel('Cluster Label', fontsize=15)
    ax[2].set_ylabel('GRDI Average', fontsize=15)
    ax[2].tick_params(axis='both', which='major', labelsize=15)

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.savefig(os.path.join(save_root, dis_no, f"cluster_{n_clusters}", 'clustering_results_map.png'), dpi=300)

In [ ]:
in_flood_no = [
    '4775',
    '4859',
]
n_clusters = 2
save_root = f'../outputs'
for dis_no in in_flood_no:
    clustering_res = read_clustering_results(dis_no, n_clusters, save_root)
    clustering_res.to_file(os.path.join(save_root, f'{dis_no}/cluster_{n_clusters}', f'clustering_result_w_grdi_{dis_no}.geojson'), driver='GeoJSON')
    viz_clustering_results(clustering_res, dis_no, n_clusters, save_root)


In [ ]:
def viz_clustering_results(clustering_res, dis_no, n_clusters=2, save_root='../outputs'):
    fig, ax = plt.subplots(1, 2, figsize=(12, 8))
    fig.suptitle(dis_no, fontsize=20)

    clustering_res.plot(column='label', ax=ax[0], legend=True, color=clustering_res['color'])
    # ax[0].set_title('Cluster Label Map', fontsize=15)
    ax[0].tick_params(axis='both', which='major', labelsize=15)

    # plot boxplot for each label
    sns.boxplot(
        x='display_label', y='grdi_avg', data=clustering_res, 
        palette=['#66a7a6', '#e16503'],
        order = ['Decrease', 'Increase']
        # palette=['#225bb2', '#60B5FF', '#FF8282', '#BB3E00'],
        # order = ['Great Decrease', 'Slight decrease', 'Slight Increase', 'Great Increase']
    ) 
    # ax[1].set_title('GRDI Average by Cluster Label', fontsize=15)
    ax[1].set_xlabel('', fontsize=15)
    ax[1].set_ylabel('GRDI', fontsize=15)
    ax[1].tick_params(axis='both', which='major', labelsize=15)
    ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=30, ha='right')

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.savefig(os.path.join(save_root, dis_no, f'cluster_{n_clusters}/{dis_no}_clustering_results_map_2.png'), dpi=300)

    plt.show()

In [ ]:
save_root = f'../outputs'
n_clusters = 2
groups = ['Decrease', 'Increase']
colors = ['#66a7a6', '#e16503']
# groups = ['Great Decrease', 'Slight decrease', 'Slight Increase', 'Great Increase']
# colors = ['#225bb2', '#60B5FF', '#FF8282', '#BB3E00']

In [ ]:
dis_no = str(4775)
save_dir = os.path.join(save_root, dis_no)
pune = gpd.read_file(
    os.path.join(save_dir, f'cluster_{n_clusters}/clustering_result_w_grdi_{dis_no}.geojson')
)

In [ ]:
# Raw cluster label integers from step 1 are not semantically stable (a rerun
# under a different sklearn could assign 0/1 the other way round), so remap to
# fixed display labels here: 0 = Decrease, 1 = Increase.
correct_label = {0: 1, 1: 0}
pune['label'] = pune['label'].map(correct_label)
pune['display_label'] = pune['label'].apply(lambda x: groups[x])
pune['color'] = pune['label'].apply(lambda x: colors[x])

# Guard against a silently inverted remap: derive each raw label's mean trend
# straight from clustering_result.csv (the only place the underlying trend
# values still live -- `pune` only carries the label) and check display label 0
# landed on the negative-trend (Decrease) cluster and display label 1 on the
# positive-trend (Increase) cluster.
_cr = pd.read_csv(os.path.join(save_dir, f'cluster_{n_clusters}', 'clustering_result.csv'))
_trend_cols = [c for c in _cr.columns if c not in ('index', 'label')]
_mean_trend = _cr.groupby('label')[_trend_cols].mean().mean(axis=1)
for _raw, _mapped in correct_label.items():
    _t = _mean_trend[_raw]
    if (_mapped == 0) != (_t < 0):
        raise ValueError(
            f"{dis_no}: correct_label={correct_label} maps raw label {_raw} "
            f"(mean trend {_t:.4f}) to display label {_mapped}, but display "
            "label 0 must be the Decrease (negative-trend) cluster and label 1 "
            "the Increase (positive-trend) cluster."
        )

pune_agg_info = pd.read_csv(os.path.join('../data', f'disaster_{dis_no}_agg_info.csv'))
pune_agg_info = pune_agg_info.drop_duplicates(subset=['geohash_5'])[['geohash_5', 'area_affected']].rename(columns={'geohash_5': 'index'}).reset_index(drop=True)
pune = pune.merge(pune_agg_info, on='index', how='left')

In [ ]:
pune.head()

In [ ]:
pune['area_affected'].value_counts()

In [ ]:
pune.to_file(os.path.join(save_dir, f'cluster_{n_clusters}/clustering_result_w_grdi_{dis_no}_correct_label.geojson'), driver='GeoJSON')

In [ ]:
viz_clustering_results(pune, dis_no, n_clusters, save_root)

In [ ]:
pune.head()

In [ ]:
dis_no = str(4859)
save_dir = os.path.join(save_root, dis_no)
clustering_res = gpd.read_file(
    os.path.join(save_root, dis_no, f'cluster_{n_clusters}/clustering_result_w_grdi_{dis_no}.geojson')
)


In [ ]:
# Raw cluster label integers from step 1 are not semantically stable (a rerun
# under a different sklearn could assign 0/1 the other way round), so remap to
# fixed display labels here: 0 = Decrease, 1 = Increase.
correct_label = {0: 0, 1: 1}
clustering_res['label'] = clustering_res['label'].map(correct_label)
clustering_res['display_label'] = clustering_res['label'].apply(lambda x: groups[x])
clustering_res['color'] = clustering_res['label'].apply(lambda x: colors[x])

# Guard against a silently inverted remap: derive each raw label's mean trend
# straight from clustering_result.csv (the only place the underlying trend
# values still live -- `clustering_res` only carries the label) and check
# display label 0 landed on the negative-trend (Decrease) cluster and display
# label 1 on the positive-trend (Increase) cluster.
_cr = pd.read_csv(os.path.join(save_dir, f'cluster_{n_clusters}', 'clustering_result.csv'))
_trend_cols = [c for c in _cr.columns if c not in ('index', 'label')]
_mean_trend = _cr.groupby('label')[_trend_cols].mean().mean(axis=1)
for _raw, _mapped in correct_label.items():
    _t = _mean_trend[_raw]
    if (_mapped == 0) != (_t < 0):
        raise ValueError(
            f"{dis_no}: correct_label={correct_label} maps raw label {_raw} "
            f"(mean trend {_t:.4f}) to display label {_mapped}, but display "
            "label 0 must be the Decrease (negative-trend) cluster and label 1 "
            "the Increase (positive-trend) cluster."
        )

agg_info = pd.read_csv(os.path.join('../data', f'disaster_{dis_no}_agg_info.csv'))
agg_info = agg_info.drop_duplicates(subset=['geohash_5'])[['geohash_5', 'area_affected']].rename(columns={'geohash_5': 'index'}).reset_index(drop=True)
clustering_res = clustering_res.merge(agg_info, on='index', how='left')

clustering_res.to_file(os.path.join(save_dir, f'cluster_{n_clusters}/clustering_result_w_grdi_{dis_no}_correct_label.geojson'), driver='GeoJSON')

In [ ]:
agg_info['area_affected'].value_counts()

In [ ]:
clustering_res['area_affected'].value_counts()


In [ ]:
viz_clustering_results(clustering_res, dis_no, n_clusters, save_root)
